# Assignment 08: Softmax, Autoregressive RNNs, and NER

This notebook is a simplified, Colab-friendly solution for the assignment. It includes:

- Part A: softmax calculations and explanations
- Part B: an autoregressive character-level GRU for next-character prediction
- Part C: text generation with greedy decoding and temperature sampling
- Part D: a small BIO-tagged NER experiment with a bidirectional LSTM
- Part E: comparison table and follow-up answers

The notebook is intentionally lightweight:

- it uses a reduced sample from `StateGrants.csv`,
- it trains for a small number of epochs by default,
- it uses a small manually created NER dataset instead of a heavy external dataset.


In [ ]:
import random
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 160)

SEED = 42
USE_REDUCED_DATA = True
FAST_DEV_RUN = False

MAX_CHAR_ROWS = 5000
CHAR_EPOCHS = 8 if not USE_REDUCED_DATA else 5
NER_EPOCHS = 10 if not USE_REDUCED_DATA else 6
CHAR_BATCH_SIZE = 64
NER_BATCH_SIZE = 32

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("Device:", DEVICE)
print("USE_REDUCED_DATA:", USE_REDUCED_DATA)


## Part A. Softmax From Logits to Probabilities


In [ ]:
logits = torch.tensor([2.0, 1.0, 0.1])

exp_logits = torch.exp(logits)
denominator = exp_logits.sum()
manual_probs = exp_logits / denominator
torch_probs = torch.softmax(logits, dim=0)

print("logits:", logits)
print("exp(logits):", exp_logits)
print("denominator:", denominator)
print("manual softmax probabilities:", manual_probs)
print("sum of probabilities:", manual_probs.sum())
print("torch.softmax(logits, dim=0):", torch_probs)


In [ ]:
large_logits = torch.tensor([1000.0, 999.0, 998.0])
shifted = large_logits - large_logits.max()
stable_probs = torch.exp(shifted) / torch.exp(shifted).sum()

print("large_logits:", large_logits)
print("shifted logits:", shifted)
print("stable probabilities:", stable_probs)
print("sum:", stable_probs.sum())

print("\nAnswers:")
print("1. We subtract the maximum logit to avoid overflow when exponentiating large numbers.")
print("2. Subtracting the same constant from every logit does not change the final probabilities.")
print("3. The first class has the highest probability because it still has the largest logit after shifting.")


In [ ]:
logits = torch.tensor([[2.0, 1.0, 0.1]])
true_class = torch.tensor([0])
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, true_class)

print("logits:", logits)
print("true_class:", true_class)
print("cross-entropy loss:", loss.item())

print("\nAnswers:")
print("1. CrossEntropyLoss expects raw logits because it internally applies a numerically stable log-softmax.")
print("2. When the model gives higher probability to the true class, the loss gets smaller.")
print("3. When the model is confident but wrong, the loss becomes large.")


## Part B. Autoregressive Character RNN


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted.")
except Exception:
    print("Colab Drive mount skipped.")

STATE_GRANTS_PATH = "/content/drive/My Drive/StateGrants.csv"

def read_stategrants_csv(default_path, local_fallback="StateGrants.csv"):
    if Path(default_path).exists():
        return pd.read_csv(default_path)
    return pd.read_csv(local_fallback)

state_grants_df = read_stategrants_csv(STATE_GRANTS_PATH)
print("Raw shape:", state_grants_df.shape)
print("Columns:", list(state_grants_df.columns))

def detect_text_column(df):
    candidates = ["name", "grant_name", "title", "program", "specialty", "speciality", "text"]
    for col in candidates:
        if col in df.columns:
            return col

    object_columns = [col for col in df.columns if df[col].dtype == object]
    if object_columns:
        return object_columns[0]
    raise ValueError(f"Could not find a text column. Columns: {list(df.columns)}")

def clean_name(text):
    text = str(text).lower().strip()
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("_", " ")
    text = re.sub(r"[^\w\s\-']", "", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

text_col = detect_text_column(state_grants_df)
raw_names = state_grants_df[text_col].dropna().astype(str).tolist()
cleaned_names = [clean_name(name) for name in raw_names]
cleaned_names = [
    name for name in cleaned_names
    if name and len(name) >= 3 and len(name) <= 40 and re.search(r"[a-zA-Zа-яА-Яәіңғүұқөһ]", name)
]
cleaned_names = sorted(set(cleaned_names))

if USE_REDUCED_DATA and len(cleaned_names) > MAX_CHAR_ROWS:
    cleaned_names = random.sample(cleaned_names, MAX_CHAR_ROWS)

if FAST_DEV_RUN:
    cleaned_names = cleaned_names[:1000]

train_names, val_names = train_test_split(cleaned_names, test_size=0.1, random_state=SEED)

print("Detected text column:", text_col)
print("Total cleaned strings:", len(cleaned_names))
print("Train strings:", len(train_names))
print("Validation strings:", len(val_names))
display(pd.DataFrame({"sample_strings": cleaned_names[:10]}))


In [ ]:
PAD_TOKEN = "<PAD>"
START_TOKEN = "<S>"
END_TOKEN = "<F>"
UNK_TOKEN = "<UNK>"

chars = sorted(set("".join(train_names)))
char2idx = {PAD_TOKEN: 0, START_TOKEN: 1, END_TOKEN: 2, UNK_TOKEN: 3}
for ch in chars:
    if ch not in char2idx:
        char2idx[ch] = len(char2idx)
idx2char = {i: ch for ch, i in char2idx.items()}

PAD_IDX = char2idx[PAD_TOKEN]
START_IDX = char2idx[START_TOKEN]
END_IDX = char2idx[END_TOKEN]
UNK_IDX = char2idx[UNK_TOKEN]

print("Character vocabulary size:", len(char2idx))

def encode_char_sequence(text):
    input_tokens = [START_TOKEN] + list(text)
    target_tokens = list(text) + [END_TOKEN]
    input_ids = [char2idx.get(token, UNK_IDX) for token in input_tokens]
    target_ids = [char2idx.get(token, UNK_IDX) for token in target_tokens]
    return input_ids, target_ids

class CharSequenceDataset(Dataset):
    def __init__(self, strings):
        self.strings = strings

    def __len__(self):
        return len(self.strings)

    def __getitem__(self, idx):
        return encode_char_sequence(self.strings[idx])

def char_collate_fn(batch):
    x_seqs, y_seqs = zip(*batch)
    max_len = max(len(seq) for seq in x_seqs)
    x_batch = torch.full((len(batch), max_len), PAD_IDX, dtype=torch.long)
    y_batch = torch.full((len(batch), max_len), PAD_IDX, dtype=torch.long)

    for i, (x_seq, y_seq) in enumerate(zip(x_seqs, y_seqs)):
        x_batch[i, :len(x_seq)] = torch.tensor(x_seq, dtype=torch.long)
        y_batch[i, :len(y_seq)] = torch.tensor(y_seq, dtype=torch.long)

    return x_batch, y_batch

train_char_ds = CharSequenceDataset(train_names)
val_char_ds = CharSequenceDataset(val_names)

train_char_loader = DataLoader(
    train_char_ds,
    batch_size=CHAR_BATCH_SIZE,
    shuffle=True,
    collate_fn=char_collate_fn,
)
val_char_loader = DataLoader(
    val_char_ds,
    batch_size=CHAR_BATCH_SIZE,
    shuffle=False,
    collate_fn=char_collate_fn,
)

x_batch, y_batch = next(iter(train_char_loader))
print("x_batch shape:", tuple(x_batch.shape))
print("y_batch shape:", tuple(y_batch.shape))
print("Example input ids:", x_batch[0][:20].tolist())
print("Example target ids:", y_batch[0][:20].tolist())


In [ ]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded, hidden)
        logits = self.fc(output)
        return logits, hidden

def char_token_accuracy(logits, targets, pad_idx):
    preds = logits.argmax(dim=-1)
    mask = targets != pad_idx
    correct = ((preds == targets) & mask).sum().item()
    total = mask.sum().item()
    return correct / max(total, 1)

def evaluate_char_model(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    batches = 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            logits, _ = model(x_batch)
            loss = criterion(logits.reshape(-1, len(char2idx)), y_batch.reshape(-1))
            total_loss += loss.item()
            total_acc += char_token_accuracy(logits, y_batch, PAD_IDX)
            batches += 1
    return total_loss / max(batches, 1), total_acc / max(batches, 1)

def train_char_model(model, train_loader, val_loader, epochs):
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    best_state = None
    best_val_loss = float("inf")
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0.0
        batches = 0

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            logits, _ = model(x_batch)
            loss = criterion(logits.reshape(-1, len(char2idx)), y_batch.reshape(-1))
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            batches += 1

        avg_train_loss = total_train_loss / max(batches, 1)
        val_loss, val_acc = evaluate_char_model(model, val_loader, criterion)
        history.append(
            {
                "epoch": epoch,
                "train_loss": round(avg_train_loss, 4),
                "val_loss": round(val_loss, 4),
                "val_next_char_acc": round(val_acc, 4),
            }
        )
        print(history[-1])

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return pd.DataFrame(history)

char_model = CharRNN(
    vocab_size=len(char2idx),
    embed_dim=32,
    hidden_size=64,
    pad_idx=PAD_IDX,
).to(DEVICE)

char_history_df = train_char_model(
    char_model,
    train_char_loader,
    val_char_loader,
    epochs=2 if FAST_DEV_RUN else CHAR_EPOCHS,
)
display(char_history_df)

plt.figure(figsize=(7, 4))
plt.plot(char_history_df["epoch"], char_history_df["train_loss"], marker="o", label="train loss")
plt.plot(char_history_df["epoch"], char_history_df["val_loss"], marker="o", label="val loss")
plt.title("Character model loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
def decode_char_ids(ids):
    tokens = [idx2char[idx] for idx in ids]
    return tokens

char_model.eval()
sample_x, sample_y = next(iter(val_char_loader))
with torch.no_grad():
    sample_logits, _ = char_model(sample_x.to(DEVICE))
sample_preds = sample_logits.argmax(dim=-1).cpu()

row_idx = 0
valid_positions = sample_y[row_idx] != PAD_IDX
input_tokens = decode_char_ids(sample_x[row_idx][valid_positions].tolist())
target_tokens = decode_char_ids(sample_y[row_idx][valid_positions].tolist())
pred_tokens = decode_char_ids(sample_preds[row_idx][valid_positions].tolist())

print("Input characters:")
print(input_tokens)
print("\nTarget characters:")
print(target_tokens)
print("\nPredicted characters:")
print(pred_tokens)


## Part C. Generate Text Autoregressively


In [ ]:
def sample_from_logits(last_logits, temperature=1.0, greedy=False):
    filtered_logits = last_logits.clone()
    filtered_logits[PAD_IDX] = -1e9
    filtered_logits[START_IDX] = -1e9
    filtered_logits[UNK_IDX] = -1e9

    if greedy:
        return int(filtered_logits.argmax().item())

    temperature = max(temperature, 1e-5)
    probs = torch.softmax(filtered_logits / temperature, dim=-1)
    return int(torch.multinomial(probs, num_samples=1).item())

def generate(start_text="", max_len=30, temperature=1.0, greedy=False):
    char_model.eval()
    prefix = clean_name(start_text)
    hidden = None

    with torch.no_grad():
        context = [START_TOKEN] + list(prefix)
        logits = None
        for token in context:
            token_id = char2idx.get(token, UNK_IDX)
            x = torch.tensor([[token_id]], dtype=torch.long, device=DEVICE)
            logits, hidden = char_model(x, hidden)

        generated = prefix
        for _ in range(max_len):
            next_id = sample_from_logits(logits[0, -1], temperature=temperature, greedy=greedy)
            next_token = idx2char[next_id]

            if next_token == END_TOKEN:
                break

            generated += next_token
            x = torch.tensor([[next_id]], dtype=torch.long, device=DEVICE)
            logits, hidden = char_model(x, hidden)

    return generated

prefixes = ["a", "ai", "nur", "al", "ka"]
generation_rows = []

for prefix in prefixes:
    generation_rows.append(
        {"setting": "greedy", "prefix": prefix, "generated": generate(prefix, max_len=30, greedy=True)}
    )

for prefix in prefixes:
    generation_rows.append(
        {
            "setting": "sampling temperature=0.5",
            "prefix": prefix,
            "generated": generate(prefix, max_len=30, temperature=0.5, greedy=False),
        }
    )

for prefix in prefixes:
    generation_rows.append(
        {
            "setting": "sampling temperature=1.0",
            "prefix": prefix,
            "generated": generate(prefix, max_len=30, temperature=1.0, greedy=False),
        }
    )

for prefix in prefixes:
    generation_rows.append(
        {
            "setting": "sampling temperature=1.5",
            "prefix": prefix,
            "generated": generate(prefix, max_len=30, temperature=1.5, greedy=False),
        }
    )

generation_df = pd.DataFrame(generation_rows)
display(generation_df)

print("Answers:")
print("1. When temperature increases, the probability distribution becomes flatter and generation becomes more diverse.")
print("2. Greedy decoding often repeats safe characters because it always chooses the locally most probable option.")
print("3. During free generation, errors can accumulate because the model conditions on its own previous mistakes.")
print("4. Training uses the correct previous characters, but generation feeds back the model's own predictions.")


## Part D. Try Named Entity Recognition


In [ ]:
def add_entity(tokens, tags, phrase, entity_type):
    words = phrase.split()
    tokens.extend(words)
    tags.append(f"B-{entity_type}")
    for _ in words[1:]:
        tags.append(f"I-{entity_type}")

people = [
    "Aigerim", "Nursultan", "Dana", "Timur", "Aruzhan",
    "Serik", "Madina", "Dias", "Asel", "Alina"
]
orgs = [
    "KazNU", "Astana IT University", "OpenAI", "Air Astana", "Kaspi Bank",
    "Nazarbayev University", "KBTU", "Google", "Yandex", "Samsung"
]
locations = [
    "Almaty", "Astana", "Shymkent", "Karaganda", "Atyrau",
    "Aktobe", "Taraz", "Semey", "Kazakhstan", "Turkistan"
]

def make_ner_sentence(template_id, person1, person2, org, loc):
    tokens, tags = [], []

    if template_id == 0:
        add_entity(tokens, tags, person1, "PER")
        tokens += ["studies", "at"]
        tags += ["O", "O"]
        add_entity(tokens, tags, org, "ORG")
        tokens += ["in"]
        tags += ["O"]
        add_entity(tokens, tags, loc, "LOC")

    elif template_id == 1:
        add_entity(tokens, tags, org, "ORG")
        tokens += ["opened", "an", "office", "in"]
        tags += ["O", "O", "O", "O"]
        add_entity(tokens, tags, loc, "LOC")

    elif template_id == 2:
        add_entity(tokens, tags, person1, "PER")
        tokens += ["visited"]
        tags += ["O"]
        add_entity(tokens, tags, loc, "LOC")
        tokens += ["with"]
        tags += ["O"]
        add_entity(tokens, tags, person2, "PER")

    elif template_id == 3:
        add_entity(tokens, tags, person1, "PER")
        tokens += ["works", "for"]
        tags += ["O", "O"]
        add_entity(tokens, tags, org, "ORG")

    elif template_id == 4:
        add_entity(tokens, tags, org, "ORG")
        tokens += ["invited"]
        tags += ["O"]
        add_entity(tokens, tags, person1, "PER")
        tokens += ["to"]
        tags += ["O"]
        add_entity(tokens, tags, loc, "LOC")

    elif template_id == 5:
        add_entity(tokens, tags, person1, "PER")
        tokens += ["from"]
        tags += ["O"]
        add_entity(tokens, tags, loc, "LOC")
        tokens += ["joined"]
        tags += ["O"]
        add_entity(tokens, tags, org, "ORG")

    elif template_id == 6:
        add_entity(tokens, tags, person1, "PER")
        tokens += ["met"]
        tags += ["O"]
        add_entity(tokens, tags, person2, "PER")
        tokens += ["in"]
        tags += ["O"]
        add_entity(tokens, tags, loc, "LOC")

    else:
        add_entity(tokens, tags, org, "ORG")
        tokens += ["hired"]
        tags += ["O"]
        add_entity(tokens, tags, person1, "PER")
        tokens += ["in"]
        tags += ["O"]
        add_entity(tokens, tags, loc, "LOC")

    return tokens, tags

random.seed(SEED)
ner_examples = []
for _ in range(180):
    person1 = random.choice(people)
    person2 = random.choice([p for p in people if p != person1])
    org = random.choice(orgs)
    loc = random.choice(locations)
    template_id = random.randint(0, 7)
    ner_examples.append(make_ner_sentence(template_id, person1, person2, org, loc))

if FAST_DEV_RUN:
    ner_examples = ner_examples[:80]

train_ner, val_ner = train_test_split(ner_examples, test_size=0.2, random_state=SEED)

print("NER train sentences:", len(train_ner))
print("NER validation sentences:", len(val_ner))
print("Sample tokens:", train_ner[0][0])
print("Sample tags:", train_ner[0][1])


In [ ]:
PAD_WORD = "<PAD>"
UNK_WORD = "<UNK>"
PAD_TAG = "<PAD>"

word_counter = Counter(token for tokens, _ in train_ner for token in tokens)
vocab_words = sorted(word_counter.keys())
tag_set = sorted({tag for _, tags in train_ner + val_ner for tag in tags})

word2idx = {PAD_WORD: 0, UNK_WORD: 1}
for token in vocab_words:
    if token not in word2idx:
        word2idx[token] = len(word2idx)

tag2idx = {PAD_TAG: 0}
for tag in tag_set:
    if tag not in tag2idx:
        tag2idx[tag] = len(tag2idx)

idx2tag = {idx: tag for tag, idx in tag2idx.items()}
PAD_WORD_IDX = word2idx[PAD_WORD]
UNK_WORD_IDX = word2idx[UNK_WORD]
PAD_TAG_IDX = tag2idx[PAD_TAG]

class NERDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        tokens, tags = self.examples[idx]
        x_ids = [word2idx.get(token, UNK_WORD_IDX) for token in tokens]
        y_ids = [tag2idx[tag] for tag in tags]
        return x_ids, y_ids, tokens, tags

def ner_collate_fn(batch):
    x_seqs, y_seqs, tokens_list, tags_list = zip(*batch)
    max_len = max(len(seq) for seq in x_seqs)
    x_batch = torch.full((len(batch), max_len), PAD_WORD_IDX, dtype=torch.long)
    y_batch = torch.full((len(batch), max_len), PAD_TAG_IDX, dtype=torch.long)

    for i, (x_seq, y_seq) in enumerate(zip(x_seqs, y_seqs)):
        x_batch[i, :len(x_seq)] = torch.tensor(x_seq, dtype=torch.long)
        y_batch[i, :len(y_seq)] = torch.tensor(y_seq, dtype=torch.long)

    return x_batch, y_batch, tokens_list, tags_list

train_ner_ds = NERDataset(train_ner)
val_ner_ds = NERDataset(val_ner)

train_ner_loader = DataLoader(
    train_ner_ds,
    batch_size=NER_BATCH_SIZE,
    shuffle=True,
    collate_fn=ner_collate_fn,
)
val_ner_loader = DataLoader(
    val_ner_ds,
    batch_size=NER_BATCH_SIZE,
    shuffle=False,
    collate_fn=ner_collate_fn,
)

x_batch, tag_batch, _, _ = next(iter(train_ner_loader))
print("word vocab size:", len(word2idx))
print("tag vocab size:", len(tag2idx))
print("x_batch shape:", tuple(x_batch.shape))
print("tag_batch shape:", tuple(tag_batch.shape))
print("tag2idx:", tag2idx)


In [ ]:
class BiRNNNER(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_tags, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.LSTM(
            embed_dim,
            hidden_size,
            batch_first=True,
            bidirectional=True,
        )
        self.fc = nn.Linear(hidden_size * 2, num_tags)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.rnn(embedded)
        logits = self.fc(output)
        return logits

def ner_token_accuracy(logits, targets, pad_tag_idx):
    preds = logits.argmax(dim=-1)
    mask = targets != pad_tag_idx
    correct = ((preds == targets) & mask).sum().item()
    total = mask.sum().item()
    return correct / max(total, 1)

def evaluate_ner_model(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    batches = 0

    with torch.no_grad():
        for x_batch, tag_batch, _, _ in loader:
            x_batch = x_batch.to(DEVICE)
            tag_batch = tag_batch.to(DEVICE)
            logits = model(x_batch)
            loss = criterion(logits.reshape(-1, len(tag2idx)), tag_batch.reshape(-1))
            total_loss += loss.item()
            total_acc += ner_token_accuracy(logits, tag_batch, PAD_TAG_IDX)
            batches += 1

    return total_loss / max(batches, 1), total_acc / max(batches, 1)

def train_ner_model(model, train_loader, val_loader, epochs):
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_TAG_IDX)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    best_state = None
    best_val_loss = float("inf")
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0.0
        batches = 0

        for x_batch, tag_batch, _, _ in train_loader:
            x_batch = x_batch.to(DEVICE)
            tag_batch = tag_batch.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = criterion(logits.reshape(-1, len(tag2idx)), tag_batch.reshape(-1))
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            batches += 1

        avg_train_loss = total_train_loss / max(batches, 1)
        val_loss, val_acc = evaluate_ner_model(model, val_loader, criterion)
        history.append(
            {
                "epoch": epoch,
                "train_loss": round(avg_train_loss, 4),
                "val_loss": round(val_loss, 4),
                "val_token_acc": round(val_acc, 4),
            }
        )
        print(history[-1])

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return pd.DataFrame(history)

ner_model = BiRNNNER(
    vocab_size=len(word2idx),
    embed_dim=64,
    hidden_size=64,
    num_tags=len(tag2idx),
    pad_idx=PAD_WORD_IDX,
).to(DEVICE)

ner_history_df = train_ner_model(
    ner_model,
    train_ner_loader,
    val_ner_loader,
    epochs=2 if FAST_DEV_RUN else NER_EPOCHS,
)
display(ner_history_df)

plt.figure(figsize=(7, 4))
plt.plot(ner_history_df["epoch"], ner_history_df["train_loss"], marker="o", label="train loss")
plt.plot(ner_history_df["epoch"], ner_history_df["val_loss"], marker="o", label="val loss")
plt.title("NER model loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_TAG_IDX)
final_val_loss, final_val_acc = evaluate_ner_model(ner_model, val_ner_loader, criterion)
print("Final validation token accuracy (ignoring padding):", round(final_val_acc, 4))

ner_model.eval()
shown = 0
all_examples = []

with torch.no_grad():
    for x_batch, tag_batch, tokens_list, tags_list in val_ner_loader:
        logits = ner_model(x_batch.to(DEVICE))
        preds = logits.argmax(dim=-1).cpu().numpy()

        for i in range(len(tokens_list)):
            tokens = list(tokens_list[i])
            true_tags = list(tags_list[i])
            pred_tags = [idx2tag[idx] for idx in preds[i][:len(tokens)]]
            df = pd.DataFrame(
                {
                    "token": tokens,
                    "true_tag": true_tags,
                    "predicted_tag": pred_tags,
                }
            )
            all_examples.append(df)
            shown += 1
            if shown == 5:
                break
        if shown == 5:
            break

for i, df in enumerate(all_examples, start=1):
    display(Markdown(f"### Validation sentence {i}"))
    display(df)


## Part E. Compare the Two RNN Uses


In [ ]:
comparison_df = pd.DataFrame(
    [
        {
            "Task": "Character generation",
            "Input": "previous characters",
            "Target": "next character",
            "Output shape": "(batch_size, seq_len, vocab_size)",
            "Uses softmax over": "character vocabulary",
            "Decoding method": "greedy or sampling step by step",
        },
        {
            "Task": "NER",
            "Input": "sentence tokens",
            "Target": "tag for each token",
            "Output shape": "(batch_size, seq_len, num_tags)",
            "Uses softmax over": "NER tag set",
            "Decoding method": "argmax tag per token",
        },
    ]
)
display(comparison_df)

print("Answers:")
print("1. Character generation is autoregressive because each next prediction depends on the previous characters and then on the model's own generated output.")
print("2. NER is not autoregressive here because the model predicts one tag for each token from the encoded sentence in parallel across positions.")
print("3. The final hidden state is not enough for NER because we need one output for every token, not one summary for the whole sentence.")
print("4. Both tasks use cross-entropy because both are classification problems over discrete classes at each prediction step.")
print("5. In both tasks, softmax helps us interpret logits as probabilities over possible output classes.")
